In [1]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt


## Process LLM Drug Indications

In [ ]:
out_dir = "./out"

In [2]:
llm_indications = pd.read_csv(f"{out_dir}/drugs_with_indications_LLM_cleaned_20260131.csv")
drugs_col = "merged_umls_label"

In [5]:
# Select columns
cols = [
    'merged_umls_label',
    'year',
    'application_number',
    'indications_first_sent',
    'indications_and_usage',
    'disease_from_indications'
]

df_subset = llm_indications[cols]

# Random sample of 50 (reproducible)
df_sample = df_subset.sample(n=50, random_state=42)

# Save
df_sample.to_csv(f"{out_dir}/llm_indications_sample_50.csv", index=False)

In [3]:
llm_indications_data = llm_indications[['unique_id', drugs_col, 'year','application_number', 'active_ingredients_split', 'disease_from_indications']]

In [4]:
llm_indications_data

,unique_id,merged_umls_label,year,application_number,active_ingredients_split,disease_from_indications
0,dexamethasone_ANDA088248,Dexamethasone,1983.0,ANDA088248,DEXAMETHASONE,asthma|atopic dermatitis|contact dermatitis|dr...
1,dexamethasone_NDA050592,Dexamethasone,1988.0,NDA050592,DEXAMETHASONE,steroid-responsive inflammatory ocular conditi...
2,dexamethasone_ANDA062938,Dexamethasone,1989.0,ANDA062938,DEXAMETHASONE,steroid-responsive inflammatory ocular conditi...
3,dexamethasone_ANDA064135,Dexamethasone,1995.0,ANDA064135,DEXAMETHASONE,steroid-responsive inflammatory ocular conditi...
4,dexamethasone_ANDA080399,Dexamethasone,1971.0,ANDA080399,DEXAMETHASONE,asthma|atopic dermatitis|contact dermatitis|dr...
...,...,...,...,...,...,...
18427,risedronate_sodium_hemi-pentahydrate_ANDA207516,Risedronate sodium hemi-pentahydrate,2019.0,ANDA207516,RISEDRONATE SODIUM HEMI-PENTAHYDRATE,postmenopausal osteoporosis|osteoporosis|gluco...
18428,risedronate_sodium_hemi-pentahydrate_NDA020835,Risedronate sodium hemi-pentahydrate,1998.0,NDA020835,RISEDRONATE SODIUM HEMI-PENTAHYDRATE,postmenopausal osteoporosis|osteoporosis|gluco...
18429,risedronate_sodium_hemi-pentahydrate_ANDA090886,Risedronate sodium hemi-pentahydrate,2014.0,ANDA090886,RISEDRONATE SODIUM HEMI-PENTAHYDRATE,postmenopausal osteoporosis|osteoporosis|gluco...
18430,risedronate_sodium_hemi-pentahydrate_NDA022560,Risedronate sodium hemi-pentahydrate,2010.0,NDA022560,RISEDRONATE SODIUM HEMI-PENTAHYDRATE,postmenopausal osteoporosis


In [5]:
llm_indications_data[llm_indications_data[drugs_col]=="Metformin"].year.min()

2002.0

In [7]:
llm_indications_data[llm_indications_data[drugs_col]=="Cladribine"].to_csv("out/Cladribine_LLM.csv",index=False)

In [6]:
llm_indications_data[llm_indications_data[drugs_col]=="Cladribine"].head()

,unique_id,merged_umls_label,year,application_number,active_ingredients_split,disease_from_indications
6631,cladribine_NDA022561,Cladribine,2019.0,NDA022561,CLADRIBINE,relapsing forms of multiple sclerosis|relapsin...
6632,cladribine_ANDA075405,Cladribine,2000.0,ANDA075405,CLADRIBINE,Hairy Cell Leukemia|anemia|neutropenia|thrombo...
6633,cladribine_ANDA076571,Cladribine,2004.0,ANDA076571,CLADRIBINE,Hairy Cell Leukemia|anemia|neutropenia|thrombo...
6634,cladribine_ANDA210856,Cladribine,2019.0,ANDA210856,CLADRIBINE,Hairy Cell Leukemia|anemia|neutropenia|thrombo...


### expand one disease per row

In [52]:
#1. Split disease list into one row per disease
df_expanded = (
    llm_indications_data.assign(disease=llm_indications_data['disease_from_indications'].str.split('|'))
      .explode('disease')
      .dropna(subset=['disease'])
)
# 1b. Standardize to lowercase and strip whitespace
df_expanded['disease'] = df_expanded['disease'].str.lower().str.strip()

# 2. Aggregate: earliest year + list of all associated documents
result = (
    df_expanded
    .groupby([drugs_col, 'disease'], as_index=False)
    .agg(
        earliest_year=('year', 'min'),
        documents=('application_number', lambda x: sorted(set(x)))
    )
)


In [53]:
result[result[drugs_col].str.contains("Thioridazine hydrochloride", case=False)]


,merged_umls_label,disease,earliest_year,documents
8212,Thioridazine hydrochloride,schizophrenia,1983.0,"[ANDA088004, ANDA088131, ANDA088133, ANDA08813..."


In [55]:
result[result['disease'].str.contains("schizophrenia", case=False)].head()


,merged_umls_label,disease,earliest_year,documents
41,ARIPiprazole,schizophrenia,2002.0,"[ANDA201519, ANDA202101, ANDA202102, ANDA20254..."
621,Asenapine,schizophrenia,2009.0,"[ANDA205960, NDA022117, NDA212268]"
1006,Brexpiprazole,schizophrenia,2015.0,"[ANDA213562, NDA205422]"
1312,Cariprazine,schizophrenia,2015.0,[NDA204370]
1908,Clozapine,schizophrenia,1989.0,"[ANDA090308, ANDA202873, ANDA212923, NDA019758..."


In [56]:
result.to_csv(f"{out_dir}/FDA_drug_disease_pairs_all_years.csv",index=False)

### link disease to ontology

In [57]:
import sys
sys.path.append("./entity_linking")   # adjust path to your real folder
from neural_based_nen import main

In [58]:
data_dir =  "./entity_linking/data/"

In [59]:
mapping_type = "disease"
col_to_map = "disease"
input_file = f"{out_dir}/FDA_drug_disease_pairs_all_years.csv"
output_file = f"{out_dir}/out/FDA_drug_disease_pairs_all_years_mapped_drug_disease.csv"

main(mapping_type, col_to_map, data_dir, input_file, output_file, stats_dir=None)

Input file: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drug_disease_pairs_all_years.csv


/home/sdonev/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using terminology: mondo
Using distance threshold: 9.65
Starting normalization for: DISEASE with cdist 9.65
Loading embeddings from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mondo/embeddings with prefix MONDO_emb...
Loading term-id pairs from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mondo/mondo_term_id_pairs.json...
No COMBINED embeddings file found, loading all batch files...
Loaded embeddings: torch.Size([129189, 768]), term_id_pairs: 129189, canonical mappings: 30084
Found 3467 unique terms in 'disease'


Mapping disease NER to mondo: 100%|██████████| 10470/10470 [00:00<00:00, 11699.42it/s]

Normalization time for 'disease': 0:01:23
Output saved to: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drug_disease_pairs_all_years_mapped_drug_disease.csv


### clean drug-disease FDA linked data

In [60]:
drug_mapped = pd.read_csv(input_file)
drug_mapped.shape

(10470, 4)

In [61]:
disease_mapped = pd.read_csv(output_file)[['disease_mondo_term_norm']]
disease_mapped.shape

(10470, 1)

In [62]:
df_drug_disease = pd.concat([drug_mapped, disease_mapped], axis=1)
df_drug_disease.head()

,merged_umls_label,disease,earliest_year,documents,disease_mondo_term_norm
0,(+)-Tolterodine,frequency,1998.0,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",frequency
1,(+)-Tolterodine,overactive bladder,1998.0,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",overactive bladder
2,(+)-Tolterodine,urge urinary incontinence,1998.0,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",urge urinary incontinence
3,(+)-Tolterodine,urgency,1998.0,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",urgency
4,(+-)-Milnacipran,fibromyalgia,2009.0,"['ANDA205081', 'NDA022256']",fibromyalgia


#### if a drug-disease pair appears several times, take earliest year

In [63]:
print("Before:", df_drug_disease.shape)


df_drug_disease = (
    df_drug_disease.groupby(
        ["merged_umls_label", "disease_mondo_term_norm"],
        as_index=False
    )
    .agg(
        earliest_year=("earliest_year", "min"),
        disease=("disease", lambda x: "; ".join(sorted(set(map(str, x))))),
        documents=("documents", lambda x: " ; ".join(map(str, x))),
    )
)

print("After:", df_drug_disease.shape)


Before: (10470, 5)
After: (9350, 5)


In [64]:
df_drug_disease.head()

,merged_umls_label,disease_mondo_term_norm,earliest_year,disease,documents
0,(+)-Tolterodine,frequency,1998.0,frequency,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN..."
1,(+)-Tolterodine,overactive bladder,1998.0,overactive bladder,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN..."
2,(+)-Tolterodine,urge urinary incontinence,1998.0,urge urinary incontinence,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN..."
3,(+)-Tolterodine,urgency,1998.0,urgency,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN..."
4,(+-)-Milnacipran,fibromyalgia,2009.0,fibromyalgia,"['ANDA205081', 'NDA022256']"


In [67]:
df_drug_disease.to_csv(f"{out_dir}/out/FDA_drug_disease_pairs_mapped_both.csv",index=False)